# Trainer

Simple trainer for fnl prediction

In [1]:
import os
import sys
import math
import logging
from datetime import datetime

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
sys.path.append("/users/stevensonb/Research/tools/deepsphere-cosmo-tf2")

import tensorflow as tf

tf.get_logger().setLevel(logging.ERROR)

import healpy as hp
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.callbacks import EarlyStopping, TerminateOnNaN
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.optimizers.schedules import CosineDecayRestarts

from deepsphere import HealpyGCNN
from deepsphere.healpy_layers import HealpyChebyshev, HealpyPool

import wandb
from wandb.integration.keras import WandbMetricsLogger

from mlpng import Core
from mlpng.utils import setup_logging, rmse_metrics
from mlpng.utils.dataloaders import KappaDataset

setup_logging("mlpng.notebook", level=logging.DEBUG)
logger = logging.getLogger("mlpng.notebook")

## Models

In [2]:
class EncoderBlock(tf.keras.layers.Layer):
    """Encoder block with HEALPix Chebyshev convolution and pooling."""

    def __init__(
        self,
        nside,
        npix,
        fin,
        fout,
        K,
        pool_p,
        max_batch_size,
        dropout_rate=0.1,
        activation="gelu",
        pool_type="AVG",
        **kwargs
    ):
        super().__init__(**kwargs)
        self.nside = nside

        # Select initializer based on activation
        initializer = (
            tf.keras.initializers.HeNormal()
            if activation == "relu"
            else tf.keras.initializers.GlorotNormal()
        )

        layers = [
            HealpyChebyshev(
                K=K,
                Fout=fout,
                activation=activation,
                use_bn=True,
                use_bias=True,
                initializer=initializer,
            ),
            Dropout(dropout_rate),
            HealpyPool(pool_p, pool_type),
        ]

        self.body = HealpyGCNN(
            nside=nside,
            indices=np.arange(npix),
            layers=layers,
            n_neighbors=8,
            max_batch_size=max_batch_size,
            initial_Fin=fin,
        )

    def call(self, x, training=False):
        return self.body(x, training=training)


class SplitHeadLayer(tf.keras.layers.Layer):
    """Split head with shared layers and per-shape branches."""

    def __init__(
        self,
        n_outputs,
        shared_units=64,
        shared_layers=1,
        split_units=32,
        split_layers=1,
        activation="gelu",
        dropout=0.0,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.n_outputs = n_outputs
        initializer = (
            tf.keras.initializers.HeNormal()
            if activation == "relu"
            else tf.keras.initializers.GlorotNormal()
        )

        # Shared layers
        self.shared = []
        for _ in range(shared_layers):
            self.shared.append(
                Dense(
                    shared_units, activation=activation, kernel_initializer=initializer
                )
            )
            if dropout > 0:
                self.shared.append(Dropout(dropout))

        # Per-output branches
        self.branches = []
        for i in range(n_outputs):
            branch = []
            for j in range(split_layers):
                branch.append(
                    Dense(
                        split_units,
                        activation=activation,
                        kernel_initializer=initializer,
                        name=f"split_{i}_dense_{j}",
                    )
                )
                if dropout > 0:
                    branch.append(Dropout(dropout, name=f"split_{i}_dropout_{j}"))
            branch.append(Dense(1, kernel_initializer=initializer, name=f"output_{i}"))
            self.branches.append(branch)

    def call(self, x, training=False):
        # Shared processing
        for layer in self.shared:
            x = layer(x, training=training) if isinstance(layer, Dropout) else layer(x)

        # Per-shape branches
        outputs = []
        for branch in self.branches:
            out = x
            for layer in branch:
                out = (
                    layer(out, training=training)
                    if isinstance(layer, Dropout)
                    else layer(out)
                )
            outputs.append(out)

        return tf.keras.layers.Concatenate()(outputs)


def build_encoder_model(
    nside,
    npix,
    npol,
    n_outputs,
    max_batch_size=128,
    pool_p=2,
    K=5,
    encoder_activation="gelu",
    head_activation="relu",
    pool_type="AVG",
    dropout_rate=0.0,
    shared_units=64,
    shared_layers=1,
    split_units=32,
    split_layers=2,
    head_dropout=0.0,
):
    """Build encoder model with split head for fnl prediction."""

    # Calculate encoder depth
    nside_factor = 2**pool_p
    depth = int(math.log(nside, nside_factor))
    level_nsides = [nside // (nside_factor**i) for i in range(depth + 1)]
    level_npixels = [12 * ns**2 for ns in level_nsides]
    channels = [npol] + [2 ** (i + 5) for i in range(depth + 1)]  # 32, 64, 128...

    # Build encoder
    inputs = tf.keras.Input(shape=(npix, npol), name="lensed_maps")
    x = inputs

    for i in range(depth):
        x = EncoderBlock(
            nside=level_nsides[i],
            npix=level_npixels[i],
            fin=channels[i],
            fout=channels[i + 1],
            K=K,
            pool_p=pool_p,
            max_batch_size=max_batch_size,
            dropout_rate=dropout_rate,
            activation=encoder_activation,
            pool_type=pool_type,
        )(x)

    # Split head
    x = Flatten()(x)
    outputs = SplitHeadLayer(
        n_outputs=n_outputs,
        shared_units=shared_units,
        shared_layers=shared_layers,
        split_units=split_units,
        split_layers=split_layers,
        activation=head_activation,
        dropout=head_dropout,
    )(x)

    model = tf.keras.Model(inputs, outputs, name="encoder_fnl")
    return model


print("Model definition loaded")

Model definition loaded


In [3]:
class TaskSpecificHeads(tf.keras.layers.Layer):
    """Three separate heads with increasing capacity for local, equilateral, orthogonal."""

    def __init__(self, n_outputs, task_names, activation="relu", dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.n_outputs = n_outputs
        init = (
            tf.keras.initializers.HeNormal()
            if activation == "relu"
            else tf.keras.initializers.GlorotNormal()
        )

        # Helper to build a head with given layer sizes
        def build_head(sizes, prefix, head_dropout=0.0, head_activation=None):
            layers = []

            if head_dropout > 0:
                layers.append(Dropout(head_dropout, name=f"{prefix}_dropout_{i+1}"))

            act = head_activation or activation
            for i, size in enumerate(sizes[:-1]):
                layers.append(
                    Dense(
                        size,
                        activation=act,
                        kernel_initializer=init,
                        name=f"{prefix}_dense_{i+1}",
                    )
                )
            layers.append(
                Dense(sizes[-1], kernel_initializer=init, name=f"{prefix}_output")
            )
            return layers

        self.head_0 = build_head([32, 32, 1], "head_0", dropout, activation)
        self.head_1 = build_head([128, 64, 32, 1], "head_1", dropout, activation)
        self.head_2 = build_head([64, 64, 32, 1], "head_2", dropout, activation)

    def call(self, x, training=False):
        outputs = []
        for head in [self.head_0, self.head_1, self.head_2]:
            out = x
            for layer in head:
                out = (
                    layer(out, training=training)
                    if isinstance(layer, Dropout)
                    else layer(out)
                )
            outputs.append(out)
        return tf.keras.layers.Concatenate()(outputs)


def build_task_model(
    nside,
    npix,
    npol,
    n_outputs,
    task_names,
    max_batch_size=128,
    pool_p=2,
    K=5,
    encoder_activation="gelu",
    head_activation="relu",
    pool_type="AVG",
    dropout_rate=0.0,
    head_dropout=0.0,
):
    """Build encoder with task-specific heads of different capacities."""

    # Calculate encoder depth
    nside_factor = 2**pool_p
    depth = int(math.log(nside, nside_factor))
    level_nsides = [nside // (nside_factor**i) for i in range(depth + 1)]
    level_npixels = [12 * ns**2 for ns in level_nsides]
    channels = [npol] + [2 ** (i + 5) for i in range(depth + 1)]

    # Build encoder
    inputs = tf.keras.Input(shape=(npix, npol), name="lensed_maps")
    x = inputs

    for i in range(depth):
        x = EncoderBlock(
            nside=level_nsides[i],
            npix=level_npixels[i],
            fin=channels[i],
            fout=channels[i + 1],
            K=K,
            pool_p=pool_p,
            max_batch_size=max_batch_size,
            dropout_rate=dropout_rate,
            activation=encoder_activation,
            pool_type=pool_type,
        )(x)

    # Task-specific heads with different capacities
    x = Flatten()(x)
    outputs = TaskSpecificHeads(
        n_outputs=n_outputs,
        task_names=task_names,
        activation=head_activation,
        dropout=head_dropout,
    )(x)

    model = tf.keras.Model(inputs, outputs, name="task_conditioned_encoder")
    return model

In [4]:
def create_sigma_weighted_loss(sigma_all, weight_scale=None):
    sigma_tensor = tf.constant(sigma_all, dtype=tf.float32)

    if weight_scale is None:
        weight_scale = [1.0] * len(sigma_all)
    weight_tensor = tf.constant(weight_scale, dtype=tf.float32)

    def sigma_weighted_mse(y_true, y_pred):
        squared_error = tf.square(y_true - y_pred)
        weights = weight_tensor / (sigma_tensor**2)
        weighted_loss = squared_error * weights
        return tf.reduce_mean(weighted_loss)

    return sigma_weighted_mse

## Setup and Training

In [5]:
# Initialize Core with nside 64 and all shapes
core = Core(["./settings/n64.json", "--shapes", "all", "--nsims", "1000"])
sigma_all = np.array(core.get_likelihoods(True))
custom_loss = create_sigma_weighted_loss(sigma_all)

06-Feb-26 08:37:43 - mlpng.core - DEBUG - Parsing CLI args: ['./settings/n64.json', '--shapes', 'all', '--nsims', '1000']
06-Feb-26 08:37:43 - mlpng.core - INFO - Loading settings from file './settings/n64.json'
06-Feb-26 08:37:43 - mlpng.core - DEBUG - Forcing setting 'nsims' to 1000 due to CLI
06-Feb-26 08:37:43 - mlpng.core - DEBUG - Forcing setting 'shapes' to ['all'] due to CLI
06-Feb-26 08:37:43 - mlpng.core - DEBUG - Forcing setting 'tf_cache' to True due to CLI
06-Feb-26 08:37:43 - mlpng.core - DEBUG - Forcing setting 'tf_mem_cache' to True due to CLI
06-Feb-26 08:37:43 - mlpng.core - DEBUG - Overriding cosmo param 'As' from 2.13e-09 to 2.105e-09
06-Feb-26 08:37:43 - mlpng.core - DEBUG - Overriding cosmo param 'ns' from 0.9624 to 0.965
06-Feb-26 08:37:43 - mlpng.core - DEBUG - Running with settings: 
{
  "cosmo_params": {
    "As": 2.105e-09,
    "ns": 0.965,
    "pivot_scalar": 0.05,
    "H0": 67.4,
    "ombh2": 0.0224,
    "omch2": 0.12,
    "tau": 0.054
  },
  "nsims": 1000,

In [6]:
import h5py

with h5py.File(core.file, mode="r", swmr=True, locking=False) as f:
    print(np.array(f.get("marginal_likelihoods")["unlensed"]))
    fish = f.get("fisher")["unlensed"]
    for i in fish:
        print(i, 1 / np.sqrt(np.mean(fish[i])))
    # print(np.array(1/np.sqrt(f.get("fisher")['lensed']['local'])))

[ 61.387436 288.4451    99.216125  61.389336 288.4543    99.21929
  61.38746  288.44513   99.21613   61.387012 288.443     99.2154
  61.38933  288.45392   99.21916   61.386772 288.44193   99.21504
  61.389793 288.45642   99.22      61.387962 288.44742   99.21693
  61.38779  288.4466    99.21665   61.39008  288.45773   99.220474]
equilateral 309.58586005733184
local 47.201040466117284
orthogonal 164.4364383814645


In [7]:
stop

NameError: name 'stop' is not defined

In [ ]:
print(f"Configuration:")
print(f"  nside: {core.nside}")
print(f"  npix:  {core.npix}")
print(f"  shapes: {core.shapes}")
print(f"  n_outputs: {len(core.shapes)}")
print(f"  Sigma {core.shapes}: {sigma_all}")

In [ ]:
DATA_FRACTION = 1.0
DUPLICATES = [25, 10, 2]  # Train, val, test
MAX_EPOCHS = 100
PATIENCE = 16
BATCH_SIZE = 64
DECAY_STEPS = core.total_sims * 0.8 * DATA_FRACTION // BATCH_SIZE

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
CACHE_FILE = f"/lustre/smuexa01/client/users/stevensonb/mlpng_cache_{timestamp}/"
os.makedirs(CACHE_FILE, exist_ok=True)
print(f"Timestamp: {timestamp}")
print(f"Cache directory: {CACHE_FILE}")

ds = KappaDataset.fromCore(core, x_output="lensed", y_output="fnl")

train_ds, val_ds, test_ds = ds.split(
    train_size=0.8 * DATA_FRACTION,
    val_size=0.1 * DATA_FRACTION,
    test_size=0.1 * DATA_FRACTION,
    to_tf=True,
    batch_size=BATCH_SIZE,
    duplicates=DUPLICATES,
    gen_batch_size=8,
    cache_file=CACHE_FILE,
)

In [ ]:
MODEL_TYPE = "task"
POOL_P = 2
K = 9
ENCODER_ACTIVATION = "gelu"
HEAD_ACTIVATION = "relu"
POOL_TYPE = "AVG"
DROPOUT_RATE = 0.1
HEAD_DROPOUT = 0.0

model = build_task_model(
    nside=core.nside,
    npix=core.npix,
    npol=core.npols,
    n_outputs=len(core.shapes),
    task_names=core.shapes,
    max_batch_size=BATCH_SIZE,
    pool_p=POOL_P,
    K=K,
    encoder_activation=ENCODER_ACTIVATION,
    head_activation=HEAD_ACTIVATION,
    pool_type=POOL_TYPE,
    dropout_rate=DROPOUT_RATE,
    head_dropout=HEAD_DROPOUT,
)

In [ ]:
model.summary(expand_nested=True, show_trainable=True)

In [ ]:
# Training with optimized hyperparameters from Trial 152
use_cosine_decay = False
initial_lr = 1e-5
decay_rate = 0.98
weight_decay = 1e-5

# Initialize Weights & Biases logging with all config
wandb.init(
    project="mlpng",
    entity="mlpng",
    config={
        # Data parameters
        "nside": core.nside,
        "npix": core.npix,
        "npols": core.npols,
        "shapes": core.shapes,
        "n_outputs": len(core.shapes),
        "nsims": core.total_sims,
        "fnl_min": core.fnl_min,
        "fnl_max": core.fnl_max,
        "sigma_values": sigma_all.tolist(),
        # Training parameters
        "data_fraction": DATA_FRACTION,
        "duplicates": DUPLICATES,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "batch_size": BATCH_SIZE,
        "decay_steps": DECAY_STEPS,
        # Model parameters
        "model_type": MODEL_TYPE,
        "pool_p": POOL_P,
        "K": K,
        "encoder_activation": ENCODER_ACTIVATION,
        "head_activation": HEAD_ACTIVATION,
        "pool_type": POOL_TYPE,
        "dropout_rate": DROPOUT_RATE,
        "head_dropout": HEAD_DROPOUT,
        # Optimizer parameters
        "use_cosine_decay": use_cosine_decay,
        "initial_lr": initial_lr,
        "decay_rate": decay_rate,
        "weight_decay": weight_decay,
    },
    tags=[f"nside-{core.nside}", MODEL_TYPE, *core.shapes],
)

if use_cosine_decay:
    lr_schedule = CosineDecayRestarts(
        initial_learning_rate=initial_lr,
        first_decay_steps=DECAY_STEPS,
        t_mul=2.0,
        m_mul=decay_rate,
        alpha=0.001,
    )
else:
    from tensorflow.keras.optimizers.schedules import ExponentialDecay

    lr_schedule = ExponentialDecay(
        initial_learning_rate=initial_lr,
        decay_steps=DECAY_STEPS,
        decay_rate=decay_rate,
        staircase=True,
    )

optimizer = AdamW(learning_rate=lr_schedule, weight_decay=weight_decay)

# Use sigma-weighted MSE loss (weights by inverse variance)
model.compile(
    optimizer=optimizer,
    loss=custom_loss,
    metrics=rmse_metrics(core.shapes),
)

In [ ]:
callbacks = [
    TerminateOnNaN(),
    EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True),
    WandbMetricsLogger(log_freq="epoch"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
print(f"Training complete! Best val_loss: {min(history.history['val_loss']):.6f}")

## Full Analysis

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history["loss"], label="Train", linewidth=2)
axes[0].plot(history.history["val_loss"], label="Val", linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].set_title("Training History")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale("log")

# Per-shape RMSE
for shape in core.shapes:
    train_key = f"rmse_{shape}"
    val_key = f"val_rmse_{shape}"
    if train_key in history.history:
        axes[1].plot(
            history.history[val_key], label=shape, marker="o", markersize=3, alpha=0.7
        )

axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("RMSE")
axes[1].set_title("Val RMSE per Shape")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
print("Evaluating on test set...")
test_predictions = model.predict(test_ds, verbose=0)

# Collect ground truth from test set
test_truth = []
for _, y in test_ds:
    test_truth.append(y.numpy())
test_truth = np.concatenate(test_truth, axis=0)
print(test_truth.shape)

# Replicate truth values across all output dimensions (same fnl for each shape)
nshapes = len(core.shapes)
# test_truth = np.tile(test_truth.reshape(-1, 1), (1, nshapes))

print(f"Predictions shape: {test_predictions.shape}")
print(f"Ground truth shape: {test_truth.shape}")

In [ ]:
# Per-shape evaluation and plots
nshapes = len(core.shapes)
sigma_all = core.get_likelihoods(True)
print(f"Sigma values: {sigma_all}")

fig, axes = plt.subplots(2, nshapes, figsize=(5 * nshapes, 10))

if nshapes == 1:
    axes = axes.reshape(2, 1)

for shape_idx, shape_name in enumerate(core.shapes):
    truth = test_truth[:, shape_idx]
    pred = test_predictions[:, shape_idx]
    error = pred - truth

    sigma = sigma_all[shape_idx] if len(sigma_all) > shape_idx else sigma_all[0]
    corr = np.corrcoef(truth, pred)[0, 1]
    rmse = np.sqrt(np.mean(error**2))

    # Scatter plot
    line = np.array([np.nanmin(truth), np.nanmax(truth)])
    axes[0, shape_idx].scatter(truth, pred, alpha=0.4, s=8)
    axes[0, shape_idx].plot(line, line, "r--", label="Perfect", linewidth=2)
    axes[0, shape_idx].plot(line, line + sigma, "g--", alpha=0.5)
    axes[0, shape_idx].plot(line, line - sigma, "g--", alpha=0.5)
    axes[0, shape_idx].set_xlabel("True fnl")
    axes[0, shape_idx].set_ylabel("Predicted fnl")
    axes[0, shape_idx].set_title(f"{shape_name}\nCorr: {corr:.3f}, RMSE: {rmse:.2f}")
    axes[0, shape_idx].grid(True, alpha=0.3)
    axes[0, shape_idx].legend(fontsize=9)

    # Error histogram
    axes[1, shape_idx].hist(error, bins=40, alpha=0.7, edgecolor="black")
    axes[1, shape_idx].axvline(
        sigma, color="g", linestyle="--", linewidth=2, label=f"+σ"
    )
    axes[1, shape_idx].axvline(
        -sigma, color="g", linestyle="--", linewidth=2, label=f"-σ"
    )
    axes[1, shape_idx].axvline(0, color="red", linestyle="-", linewidth=1, alpha=0.5)
    axes[1, shape_idx].set_xlabel("Prediction Error")
    axes[1, shape_idx].set_ylabel("Count")
    axes[1, shape_idx].set_title(f"Error Distribution")
    axes[1, shape_idx].grid(True, alpha=0.3, axis="y")
    axes[1, shape_idx].legend(fontsize=9)

fig.suptitle("Test Set Results - All Shapes", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Summary metrics table
import pandas as pd

summary = []
for shape_idx, shape_name in enumerate(core.shapes):
    truth = test_truth[:, shape_idx]
    pred = test_predictions[:, shape_idx]
    error = pred - truth

    sigma = sigma_all[shape_idx] if len(sigma_all) > shape_idx else sigma_all[0]

    summary.append(
        {
            "Shape": shape_name,
            "MAE": np.mean(np.abs(error)),
            "RMSE": np.sqrt(np.mean(error**2)),
            "RMSE/Sigma": np.sqrt(np.mean(error**2)) / sigma,
            "Correlation": np.corrcoef(truth, pred)[0, 1],
        }
    )

summary_df = pd.DataFrame(summary)
print("\nTest Set Performance Summary")
print("=" * 80)
print(summary_df.to_string(index=False))
print("=" * 80)

## Restricted FNL Range Analysis

In [ ]:
# Configure restricted fnl range
FNL_MIN = -100
FNL_MAX = 100

# Filter test set to restricted range - INDEPENDENTLY for each shape
# Each shape has its own mask based on its truth values
shape_masks = {}
for shape_idx, shape_name in enumerate(core.shapes):
    shape_truth = test_truth[:, shape_idx]
    shape_masks[shape_name] = (shape_truth >= FNL_MIN) & (shape_truth <= FNL_MAX)

print(f"Restricted FNL Range: [{FNL_MIN}, {FNL_MAX}]")
print(f"Sigma values: {sigma_all}")
print(f"\nSamples in range per shape:")
for shape_name, mask in shape_masks.items():
    print(
        f"  {shape_name}: {mask.sum()} / {len(test_truth)} ({100*mask.sum()/len(test_truth):.1f}%)"
    )

In [ ]:
# Summary metrics for restricted range (each shape filtered independently)
restricted_summary = []
for shape_idx, shape_name in enumerate(core.shapes):
    mask = shape_masks[shape_name]
    truth = test_truth[mask, shape_idx]
    pred = test_predictions[mask, shape_idx]
    error = pred - truth

    sigma = sigma_all[shape_idx] if len(sigma_all) > shape_idx else sigma_all[0]

    restricted_summary.append(
        {
            "Shape": shape_name,
            "N": mask.sum(),
            "MAE": np.mean(np.abs(error)),
            "RMSE": np.sqrt(np.mean(error**2)),
            "RMSE/Sigma": np.sqrt(np.mean(error**2)) / sigma,
            "Correlation": np.corrcoef(truth, pred)[0, 1] if len(truth) > 1 else np.nan,
            "Sigma": sigma,
        }
    )

restricted_df = pd.DataFrame(restricted_summary)
print(
    f"\nRestricted Range [{FNL_MIN}, {FNL_MAX}] Performance Summary (per-shape filtering)"
)
print("=" * 100)
print(restricted_df.to_string(index=False))
print("=" * 100)

# Print sigma interpretation
print("\nSigma Interpretation (measurement uncertainty from likelihood analysis):")
for row in restricted_summary:
    ratio = row["RMSE/Sigma"]
    status = "below σ" if ratio < 1.0 else "above σ"
    print(
        f"  {row['Shape']:12s}: N={row['N']:4d}, σ = {row['Sigma']:.2f}, RMSE/σ = {ratio:.3f} ({status})"
    )

In [ ]:
# Visualization for restricted range (each shape filtered independently)
nshapes = len(core.shapes)

fig, axes = plt.subplots(2, nshapes, figsize=(5 * nshapes, 10))

if nshapes == 1:
    axes = axes.reshape(2, 1)

for shape_idx, shape_name in enumerate(core.shapes):
    mask = shape_masks[shape_name]
    truth = test_truth[mask, shape_idx]
    pred = test_predictions[mask, shape_idx]
    error = pred - truth

    sigma = sigma_all[shape_idx] if len(sigma_all) > shape_idx else sigma_all[0]
    corr = np.corrcoef(truth, pred)[0, 1] if len(truth) > 1 else np.nan
    rmse = np.sqrt(np.mean(error**2))

    # Scatter plot
    line = np.array([FNL_MIN, FNL_MAX])
    axes[0, shape_idx].scatter(truth, pred, alpha=0.5, s=12)
    axes[0, shape_idx].plot(line, line, "r--", label="Perfect", linewidth=2)
    axes[0, shape_idx].plot(
        line, line + sigma, "g--", alpha=0.5, label=f"±σ ({sigma:.1f})"
    )
    axes[0, shape_idx].plot(line, line - sigma, "g--", alpha=0.5)
    axes[0, shape_idx].set_xlabel("True fnl")
    axes[0, shape_idx].set_ylabel("Predicted fnl")
    axes[0, shape_idx].set_title(
        f"{shape_name} (N={mask.sum()})\nCorr: {corr:.3f}, RMSE: {rmse:.2f}"
    )
    axes[0, shape_idx].set_xlim(FNL_MIN - 1, FNL_MAX + 1)
    axes[0, shape_idx].set_ylim(FNL_MIN - sigma - 1, FNL_MAX + sigma + 1)
    axes[0, shape_idx].grid(True, alpha=0.3)
    axes[0, shape_idx].legend(fontsize=9)

    # Error histogram
    axes[1, shape_idx].hist(error, bins=30, alpha=0.7, edgecolor="black")
    axes[1, shape_idx].axvline(
        sigma, color="g", linestyle="--", linewidth=2, label=f"+σ"
    )
    axes[1, shape_idx].axvline(
        -sigma, color="g", linestyle="--", linewidth=2, label=f"-σ"
    )
    axes[1, shape_idx].axvline(0, color="red", linestyle="-", linewidth=1, alpha=0.5)
    axes[1, shape_idx].set_xlabel("Prediction Error")
    axes[1, shape_idx].set_ylabel("Count")
    axes[1, shape_idx].set_title(f"Error Distribution (σ={sigma:.1f})")
    axes[1, shape_idx].grid(True, alpha=0.3, axis="y")
    axes[1, shape_idx].legend(fontsize=9)

fig.suptitle(
    f"Restricted Range [{FNL_MIN}, {FNL_MAX}] - Test Set Results (per-shape filtering)",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

In [ ]:
# Comparison: Full range vs Restricted range (per-shape filtering)
comparison_data = []
for shape_idx, shape_name in enumerate(core.shapes):
    # Full range metrics
    full_truth = test_truth[:, shape_idx]
    full_pred = test_predictions[:, shape_idx]
    full_error = full_pred - full_truth
    sigma = sigma_all[shape_idx] if len(sigma_all) > shape_idx else sigma_all[0]

    # Restricted range metrics (per-shape filtering)
    mask = shape_masks[shape_name]
    rest_truth = test_truth[mask, shape_idx]
    rest_pred = test_predictions[mask, shape_idx]
    rest_error = rest_pred - rest_truth

    comparison_data.append(
        {
            "Shape": shape_name,
            "Full N": len(full_truth),
            "Full RMSE": np.sqrt(np.mean(full_error**2)),
            "Full RMSE/σ": np.sqrt(np.mean(full_error**2)) / sigma,
            f"[{FNL_MIN},{FNL_MAX}] N": mask.sum(),
            f"[{FNL_MIN},{FNL_MAX}] RMSE": np.sqrt(np.mean(rest_error**2)),
            f"[{FNL_MIN},{FNL_MAX}] RMSE/σ": np.sqrt(np.mean(rest_error**2)) / sigma,
            "σ": sigma,
        }
    )

comparison_df = pd.DataFrame(comparison_data)
print("\nComparison: Full Range vs Restricted Range (per-shape filtering)")
print("=" * 120)
print(comparison_df.to_string(index=False))
print("=" * 120)

In [ ]:
# Log final metrics to W&B and finish run
final_metrics = {
    "best_val_loss": min(history.history["val_loss"]),
    "final_epoch": len(history.history["loss"]),
}

# Log full range test metrics
for row in summary:
    shape = row["Shape"]
    final_metrics[f"test/{shape}_mae"] = row["MAE"]
    final_metrics[f"test/{shape}_rmse"] = row["RMSE"]
    final_metrics[f"test/{shape}_rmse_sigma"] = row["RMSE/Sigma"]
    final_metrics[f"test/{shape}_correlation"] = row["Correlation"]

# Log restricted range test metrics
for row in restricted_summary:
    shape = row["Shape"]
    final_metrics[f"test_restricted/{shape}_mae"] = row["MAE"]
    final_metrics[f"test_restricted/{shape}_rmse"] = row["RMSE"]
    final_metrics[f"test_restricted/{shape}_rmse_sigma"] = row["RMSE/Sigma"]
    final_metrics[f"test_restricted/{shape}_correlation"] = row["Correlation"]

final_metrics["restricted_fnl_min"] = FNL_MIN
final_metrics["restricted_fnl_max"] = FNL_MAX

wandb.log(final_metrics)
wandb.finish()